# Initializing kcat values for PAMparametrizer

In this notebook, we give an overview of how an initial set of turnover number can be obtained. To enable this, we use multiple databases to map kcat values obtained from machine learning (from the [GotEnzymes](https://metabolicatlas.org/gotenzymes) database) to the proteins and reactions in the model using the gene-protein-reaction associations.

Before we start, we have to set up all the package and files required. Adjust the filenames and paths as desired. Importantly, make sure to **adjust the regular expression in the cell below** in order to find the gene names associated with the organism you want to model.

In [1]:
import pandas as pd
import os
from cobra.io import load_json_model, read_sbml_model
import matplotlib.pyplot as plt
import re
import numpy as np
import datetime

if os.path.split(os.getcwd())[1]=='i1_preprocessing':
    os.chdir('../..')
    
from PAModelpy.utils.pam_generation import merge_enzyme_complexes, get_protein_gene_mapping, _parse_gpr


DEFAULT_PROT_MW = 39959.4825 #g/mol
DEFAULT_KCAT =  13.7 #/s median value from BRENDA

PAM_DATA_FILE_PATH = os.path.join('Data', 'proteinAllocationModel_iML1515_EnzymaticData_new.xls')
UNIPROT_FILE_PATH = os.path.join('Data', 'Databases','uniprotkb_cglutamicumATCC13032_250124.xlsx')
NCBI_FILE_PATH = os.path.join('Data','Cglutanicum_phenotypes','NCBI_genes_corynglutatcc13032_250207.tsv')
GOTENZYMES_FILE_PATH = os.path.join('Data','Databases', 'GotEnzymes_cgl_250124.json')
MODEL_FILE_PATH = os.path.join('Models', 'iCGB21FR_annotated.xml')

Loading PAModelpy modules version 0.0.4.6


In [2]:
#output file path
# Create a datetime object for the current date
current_date = datetime.datetime.now()
# Format the date as yymmdd
formatted_date = current_date.strftime('%y%m%d')

OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'proteinAllocationModel_iCGB21FR_EnzymaticData_{formatted_date}.xlsx')
AE_OUTPUT_FILE_PATH = os.path.join('Results', '1_preprocessing', f'{formatted_date}_cgl_ActiveEnzymeInformation_GotEnzymes.xlsx')

Relatively complicated regex because of complex gene names

1. WP_\d+_\d+: Matches identifiers starting with WP_ followed by digits (\d+), an underscore, and more digits.
2. |: Alternation operator to separate different patterns.
3. lcl_NC_\d+_\d+_prot_: Matches strings starting with lcl_NC_, followed by digits, an underscore, more digits, _prot_, and then:

    - WP_\d+_\d+_\d+: Matches WP_ identifiers with an additional _ and digits at the end.
    - |\d+: Matches a plain numeric sequence following _prot_.

4. |: Alternation operator for the next part.
5. cgl\d{4}: Matches strings starting with cgl followed by exactly 4 digits.

In [3]:
#You need to adjust this to find the geneid/locustag for your microbe
locustag_regex =r'(?:WP_\d+_\d+|lcl_NC_\d+_\d+_prot_(?:WP_\d+_\d+_\d+|\d+)|[cC]gl\d{4}|[cC]g\d{4})'

## 0. Download information from different databases
1. **Metabolic model stoichiometries and reactions: [BioModels](https://www.ebi.ac.uk/biomodels/)**
- Get your model! Or use your own reaction identifiers
- In this example we will use the iCGB21FR model for [*Corynebacterium glutamicum* ATCC 13032](https://www.ebi.ac.uk/biomodels/MODEL2102050001#Overview)

2. **Turnover numbers: [GotEnzymes](https://metabolicatlas.org/gotenzymes)**
- Go to the [API of the Metabolic Atlas](https://metabolicatlas.org/api/v2/#/GotEnzymes/gotEnzymes)
- Use the `GET\gotenzymes\enzymes?` functionality, click `Try it out`
- Here we filled in `cgl` (*C. glutanicum* ATCC 13032) for `organism` and `10000` for `pagination[pageSize]`. The rest was left empty
- Click execute and a json file with all the information will be downloaded

3. **Gene-Protein-Reaction relations: [UniprotKB](https://www.uniprot.org/)**
- Go to the [Uniprot knowledge base](https://www.uniprot.org/) and used advanced search to query for your organism. For *C. glutanicum* we used [this](https://www.uniprot.org/uniprotkb?query=(taxonomy_id:196627)) webpage
- Click download, select `format`:`Excel` and select the following: 
    - `Uniprot Data -> Names & Taxonomy`: `Gene Names`
    - `Uniprot Data -> Sequences`: `length`, `mass`
    - `Uniprot Data -> Function`: `RheaID`
- Download the resulting Excel file

4. **NCBI gene id to KEGG gene id mapping**: [NCBI database](https://www.ncbi.nlm.nih.gov/)
- The model identifiers for this model are in NCBI format, but we need the KEGG locus tag ids to map the genes to the corresponding Uniprot ids. Therefore, we use the mapping provided by NCBI.
- Go to the NCBI gene list for *C. glutanicum* on [NCBI genes](https://www.ncbi.nlm.nih.gov/gene/?term=Corynebacterium+glutamicum+ATCC+13032)
- On the top left click `send to`, select `file` and `text`

## 1. Get information from the model

In the metabolic model itself is already quite some information stored. In order to obtain up-to-date information on the enzymes related to the reaction, we will need to obtain the kegg reaction ID, the reactants, the product and the gene-protein relation.

In [27]:
def create_id_mapper_from_model(model):
    expand=False

    #make a id mapping and create a mapping dataframe
    id_mapper_df = pd.DataFrame(columns = ['rxn_id', 'kegg_id', 'rhea_id', 'Reactants','Products','EC', 'GPR'])
    for rxn in model.reactions:
        #skip transport reactions and ATP maintenance requirement
        if 'ex' in rxn.id or 'EX' in rxn.id or rxn.id == 'ATPM':
            continue
        to_append= []   

        to_append.append(rxn.id)
        #not all reactions are annotated with a KEGG ID in the model
        try: 
            to_append.append(rxn.annotation['kegg.reaction'])
        except:
            to_append.append(np.nan)
        try: 
            to_append.append(rxn.annotation['rhea'])
        except:
            to_append.append(np.nan)
        try:
            to_append.append([reac.annotation['kegg.compound'] for reac in rxn.reactants])
        except:
            to_append.append([])
        try: 
            to_append.append([prod.annotation['kegg.compound'] for prod in rxn.products])
        except:
            to_append.append([])
        if 'ec-code' in rxn.annotation.keys():
            #if there are multiple enzymes make seperate lists
            if isinstance(rxn.annotation['ec-code'], list) and expand:
                info = to_append.copy()
                for i in range(len(rxn.annotation['ec-code'])):
                    #make a new row for each enzyme by copying the previous information
                    info = to_append.copy()
                    ec= rxn.annotation['ec-code'][i]
                    info.append(ec)
                    #add gpr
                    info.append(rxn.gene_reaction_rule)
                    #add to df
                    id_mapper_df.loc[len(id_mapper_df)] = info
            else:
                to_append.append(rxn.annotation['ec-code'])
                to_append.append(rxn.gene_reaction_rule)
                id_mapper_df.loc[len(id_mapper_df)] = to_append
        else:
            to_append.append(np.nan)
            #ignore entries without enzyme and gpr relation
            if rxn.gene_reaction_rule =='':
                continue
            else:
                to_append.append(rxn.gene_reaction_rule)
            id_mapper_df.loc[len(id_mapper_df)] = to_append

    #remove entries without GRP relation or Ec number


    print('Number of reactions in the mapping dataframe: ',len(id_mapper_df))
    print('Number of reactions mapped to KEGG identifier: ', len(id_mapper_df.kegg_id.dropna()))
    return id_mapper_df

def create_gene_id_mapper_df(model):
    mapper_df = pd.DataFrame(columns = ['rxn_id', 'gene_id', 'uniprot_id', 'kegg_gene', 'NCBI_protein'])
    for rxn in model.reactions:
        for gene in rxn.genes:
            row = [rxn.id, gene.id]
            for dbxref in ['uniprot', 'kegg.genes','ncbiprotein']:
                if dbxref in gene.annotation:
                    row.append(gene.annotation[dbxref])
                else:
                    row.append(np.nan)
            mapper_df.loc[len(mapper_df)] = row
            
    #need to split of cgb: in front of kegg.gene id
    mapper_df['kegg_gene'] = mapper_df['kegg_gene'].apply(
    lambda gene: gene.split(':')[1] if isinstance(gene, str) and ':' in gene else gene
    )    
    print('Number of genes in the mapping dataframe: ',len(mapper_df))
    print('Number of genes mapped to KEGG identifier: ', len(mapper_df.kegg_gene.dropna()))
    print('Number of genes mapped to uniprot identifier: ', len(mapper_df.uniprot_id.dropna()))
    return mapper_df
            

print('mapping the ids from the iCGB21FR model')
model =read_sbml_model(MODEL_FILE_PATH)
id_mapper = create_id_mapper_from_model(model)
print('\nmapping gene ids')
gene_id_mapper = create_gene_id_mapper_df(model)
id_mapper

mapping the ids from the iCGB21FR model
Number of reactions in the mapping dataframe:  1152
Number of reactions mapped to KEGG identifier:  531

mapping gene ids
Number of genes in the mapping dataframe:  1659
Number of genes mapped to KEGG identifier:  1516
Number of genes mapped to uniprot identifier:  1595


,rxn_id,kegg_id,rhea_id,Reactants,Products,EC,GPR
0,2AGPEAT160,NaN,NaN,[],[],2.3.1.40,
1,2AGPEAT180,NaN,NaN,[],[],2.3.1.40,
2,2MAHMP,NaN,"[27915, 27914, 27916, 27917]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or lc...
3,3HAD140,R04568,NaN,[C04688],"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[4.2.1.61, 2.3.1.86, 2.3.1.-, 2.3.1.85, 4.2.1.59]",
4,3MBt2pp,NaN,NaN,"[C00080, C08262]","[C08262, C00080]",NaN,lcl_NC_006958_1_prot_WP_011013917_1_810
...,...,...,...,...,...,...,...
1147,Ca2t2,NaN,NaN,"[C00076, C00080]","[C00076, C00080]",NaN,WP_011014107_1
1148,CAT,R00009,"[20309, 20310, 20311, 20312]","[[C00027, C00027;D00008]]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[1.11.1.21, 1.11.1.6]",WP_011013509_1
1149,SUCDi,NaN,"[29187, 29188, 29189, 29190]",[],[],NaN,WP_011013606_1 and WP_011013607_1
1150,PRFGS_1,R04463,"[17129, 17131, 17130, 17132]","[[C00002, C00002;D08646], C04376, [C00064, C00...","[[C00008, C00008;G11113], C04640, [C00025, C00...",6.3.5.3,lcl_NC_006958_1_prot_WP_003854010_1_2470


In [28]:
gene_id_mapper

,rxn_id,gene_id,uniprot_id,kegg_gene,NCBI_protein
0,2MAHMP,lcl_NC_006958_1_prot_WP_011015468_1_2761,Q8NLP8,cg3199,WP_011015468.1
1,2MAHMP,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,NaN,WP_003855288.1
2,3MBt2pp,lcl_NC_006958_1_prot_WP_011013917_1_810,Q8NS49,cg0953,WP_011013917.1
3,3MBt4pp,lcl_NC_006958_1_prot_WP_011013917_1_810,Q8NS49,cg0953,WP_011013917.1
4,3OXCOAT,lcl_NC_006958_1_prot_WP_003859251_1_2275,Q8NN21,cg2625,WP_003859251.1
...,...,...,...,...,...
1654,SUCDi,WP_011013607_1,NaN,NaN,NaN
1655,SUCDi,WP_011013606_1,NaN,NaN,NaN
1656,PRFGS_1,lcl_NC_006958_1_prot_WP_003854010_1_2470,Q8NMI3,cg2865,WP_003854010.1
1657,PYRt,WP_011014733_1,NaN,NaN,NaN


### Map the NCBI gene names to KEGG gene names

In the iCGB21FR model, the gene names are by default the ncbi gene names. To map the gene-protein-reaction associations to the corresponding uniprot peptide identifiers, we need to map the model gene names to the kegg gene names.

In [33]:
gene_mapper = gene_id_mapper[['gene_id', 'kegg_gene']].set_index('gene_id').to_dict()['kegg_gene']
gene_mapper

{'lcl_NC_006958_1_prot_WP_011015468_1_2761': 'cg3199',
 'lcl_NC_006958_1_prot_WP_003855288_1_2949': nan,
 'lcl_NC_006958_1_prot_WP_011013917_1_810': 'cg0953',
 'lcl_NC_006958_1_prot_WP_003859251_1_2275': 'cg2625',
 'lcl_NC_006958_1_prot_WP_011015386_1_2669': 'cg3096',
 'lcl_NC_006958_1_prot_WP_003859243_1_2278': 'cg2628',
 'lcl_NC_006958_1_prot_WP_011265680_1_1044': 'cg1226',
 'lcl_NC_006958_1_prot_WP_003857140_1_156': 'cg0196',
 'lcl_NC_006958_1_prot_WP_011014966_1_2112': nan,
 'lcl_NC_006958_1_prot_WP_011015481_1_2778': 'cg3216',
 'lcl_NC_006958_1_prot_WP_003859586_1_2138': 'cg2473',
 'lcl_NC_006958_1_prot_WP_011013682_1_467': 'cg0566',
 'lcl_NC_006958_1_prot_WP_011015122_1_2318': 'cg2680',
 'lcl_NC_006958_1_prot_WP_011013720_1_527': 'cg0637',
 'lcl_NC_006958_1_prot_WP_011013661_1_441': 'cg0535',
 'lcl_NC_006958_1_prot_WP_011015295_1_2545': 'cg2953',
 'lcl_NC_006958_1_prot_WP_011014125_1_1074': 'cg1257',
 'lcl_NC_006958_1_prot_WP_080558827_1_1134': 'cg1321',
 'lcl_NC_006958_1_prot_WP

In [40]:
#replace the ncbi gene names in the GPR relation
def replace_ids(gpr, mapper):
    if pd.isna(gpr):  # Handle missing values
        return gpr
    for old_id, new_id in mapper.items():
        if pd.isna(new_id): continue
        gpr = gpr.replace(old_id, new_id)
    return gpr

id_mapper_gpr = id_mapper.copy()
# Apply the replacement function to the GPR column
id_mapper_gpr["GPR_old"] = id_mapper_gpr["GPR"].copy()
id_mapper_gpr["GPR"] = id_mapper_gpr["GPR"].apply(lambda x: replace_ids(x, gene_mapper))
id_mapper_gpr["GPR_old"] = id_mapper_gpr["GPR"].apply(lambda x: replace_ids(x, gene_mapper))

id_mapper_gpr

,rxn_id,kegg_id,rhea_id,Reactants,Products,EC,GPR,GPR_old
0,2AGPEAT160,NaN,NaN,[],[],2.3.1.40,,
1,2AGPEAT180,NaN,NaN,[],[],2.3.1.40,,
2,2MAHMP,NaN,"[27915, 27914, 27916, 27917]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...
3,3HAD140,R04568,NaN,[C04688],"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[4.2.1.61, 2.3.1.86, 2.3.1.-, 2.3.1.85, 4.2.1.59]",,
4,3MBt2pp,NaN,NaN,"[C00080, C08262]","[C08262, C00080]",NaN,cg0953,cg0953
...,...,...,...,...,...,...,...,...
1147,Ca2t2,NaN,NaN,"[C00076, C00080]","[C00076, C00080]",NaN,WP_011014107_1,WP_011014107_1
1148,CAT,R00009,"[20309, 20310, 20311, 20312]","[[C00027, C00027;D00008]]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[1.11.1.21, 1.11.1.6]",WP_011013509_1,WP_011013509_1
1149,SUCDi,NaN,"[29187, 29188, 29189, 29190]",[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013606_1 and WP_011013607_1
1150,PRFGS_1,R04463,"[17129, 17131, 17130, 17132]","[[C00002, C00002;D08646], C04376, [C00064, C00...","[[C00008, C00008;G11113], C04640, [C00025, C00...",6.3.5.3,cg2865,cg2865


## 2. Get all the protein information

Besides information about the reactions, we need the relation to the proteins in order to establish a protein allocation model. We do this using the gene-protein-reaction association obtained from the BIGG model. The genes is the model can be mapped to the Uniprot accessions, which can be mapped to the KEGG reactions using the Rhea database.

Parse the BIGG gene-protein-reaction associations

In [51]:
#You need to adjust this to find the geneid/locustag for your microbe
#locustag_regex =r'\b([b|s]\d{4})\b'
def extract_b_numbers(text):
    if pd.isna(text):
        return text
    return re.findall(locustag_regex, text)

id_mapper_df = id_mapper_gpr.copy()

id_mapper_df['b_number'] = id_mapper_df['GPR'].apply(extract_b_numbers)
id_mapper_df['gene_id_old'] = id_mapper_df['GPR_old'].apply(extract_b_numbers)
id_mapper_df = id_mapper_df.explode(['b_number', 'gene_id_old'], ignore_index=True)
id_mapper_df = id_mapper_df.explode('EC', ignore_index=True)
id_mapper_df = id_mapper_df.merge(
    gene_id_mapper[['uniprot_id', 'gene_id']], left_on='gene_id_old', right_on='gene_id'
).drop(['gene_id_old', 'GPR_old'], axis=1)
id_mapper_df

,rxn_id,kegg_id,rhea_id,Reactants,Products,EC,GPR,b_number,uniprot_id,gene_id
0,2MAHMP,NaN,"[27915, 27914, 27916, 27917]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949
1,2MAHMP,NaN,"[27915, 27914, 27916, 27917]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949
2,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.-,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949
3,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.-,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949
4,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.15,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949
...,...,...,...,...,...,...,...,...,...,...
688,CAT,R00009,"[20309, 20310, 20311, 20312]","[[C00027, C00027;D00008]]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",1.11.1.6,WP_011013509_1,WP_011013509_1,NaN,WP_011013509_1
689,SUCDi,NaN,"[29187, 29188, 29189, 29190]",[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013606_1,NaN,WP_011013606_1
690,SUCDi,NaN,"[29187, 29188, 29189, 29190]",[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013607_1,NaN,WP_011013607_1
691,PYRt,NaN,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1,NaN,WP_011014733_1


In [56]:
def extract_kegg_gene_ids(text):
    if pd.isna(text):
        return text
    return re.findall(r'Cgl\d{4}', text)

# load the uniprot information
uniprot_df = pd.read_excel(UNIPROT_FILE_PATH)
#get the gene id from the gene names
uniprot_df['b_number'] = uniprot_df['Gene Names'].apply(extract_b_numbers)
# uniprot_df['kegg_gene'] = uniprot_df['Gene Names'].apply(extract_kegg_gene_ids)
uniprot_df = uniprot_df.explode('b_number', ignore_index=True)
# uniprot_df = uniprot_df.explode('kegg_gene', ignore_index=True)

uniprot_df = uniprot_df.drop(['Gene Names'], axis=1)
uniprot_df_genemapper = uniprot_df[['Entry', 'Mass', 'Length']]


### 2.3 Match Uniprot to BIGG data

In [61]:
# Merge first on b_number
id_mapper_with_protein = pd.merge(
    id_mapper_df, 
    uniprot_df_genemapper, 
    left_on='uniprot_id', 
    right_on = 'Entry',
    how='left'
)

# Combine the two merges: prefer the first merge, fill missing values from the second
id_mapper_with_protein['Entry'] = id_mapper_with_protein['Entry'].fillna(fallback_merge['Entry'])
id_mapper_with_protein = id_mapper_with_protein.drop('Entry', axis=1)
id_mapper_with_protein

,rxn_id,kegg_id,rhea_id,Reactants,Products,EC,GPR,b_number,uniprot_id,gene_id,Mass,Length
0,2MAHMP,NaN,"[27915, 27914, 27916, 27917]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949,36654.0,322.0
1,2MAHMP,NaN,"[27915, 27914, 27916, 27917]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949,36654.0,322.0
2,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.-,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949,36654.0,322.0
3,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.-,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949,36654.0,322.0
4,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.15,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949,36654.0,322.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1176,CAT,R00009,"[20309, 20310, 20311, 20312]","[[C00027, C00027;D00008]]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",1.11.1.6,WP_011013509_1,WP_011013509_1,NaN,WP_011013509_1,NaN,NaN
1177,SUCDi,NaN,"[29187, 29188, 29189, 29190]",[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013606_1,NaN,WP_011013606_1,NaN,NaN
1178,SUCDi,NaN,"[29187, 29188, 29189, 29190]",[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013607_1,NaN,WP_011013607_1,NaN,NaN
1179,PYRt,NaN,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1,NaN,WP_011014733_1,NaN,NaN


## 3. Match the BiGG model reaction and enzymes to the kcats from GotEnzymes
Find first all enzymes for a reaction, then for each enzyme determine if the kcats from GotEnzymes are for a forward or reverse reaction

In [52]:
id_mapper_final = id_mapper_df.copy()
id_mapper_final = id_mapper_final.rename({'uniprot_id':'enzyme_id'}, axis=1)
id_mapper_final = id_mapper_final.explode('kegg_id', ignore_index=True)

# id_mapper_final = id_mapper_final.drop_duplicates(subset = ['kegg_id'])
id_mapper_final.head()

,rxn_id,kegg_id,rhea_id,Reactants,Products,EC,GPR,b_number,enzyme_id,gene_id
0,2MAHMP,NaN,"[27915, 27914, 27916, 27917]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949
1,2MAHMP,NaN,"[27915, 27914, 27916, 27917]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949
2,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.-,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949
3,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.-,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949
4,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.15,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949


In [53]:
#get the json file from GotEnzymes and parse into workable format
#read in the json file obtained from gotEnzymes (for 'eco')
eco_enzymes_json = pd.read_json(GOTENZYMES_FILE_PATH)
#convert to a readible format
eco_enzymes = eco_enzymes_json.enzymes.to_list()
eco_enzymes_df = pd.DataFrame(eco_enzymes)

#ensure there is only one ec number per row
eco_enzymes_df['ec_number'] = eco_enzymes_df['ec_number'].str.split(pat=';')
eco_enzymes_df = eco_enzymes_df.explode('ec_number', ignore_index=True)
eco_enzymes_df = eco_enzymes_df.dropna(axis=0, subset=['ec_number'])
eco_enzymes_df = eco_enzymes_df.drop(['organism', 'domain'], axis = 1)
eco_enzymes_df.head()

,gene,reaction_id,ec_number,compound,kcat_values
0,Cgl0023,R01664,3.1.3.5,C00239,5.4287
1,Cgl0023,R01664,3.1.3.89,C00239,5.4287
2,Cgl0023,R01664,3.1.3.-,C00239,5.4287
3,Cgl0023,R00511,3.1.3.5,C00475,5.9248
4,Cgl0023,R00511,3.1.3.91,C00475,5.9248


In [54]:
# match BiGG and GotEnzymes based on gene id
eco_enzymes_merged = id_mapper_final.merge(eco_enzymes_df, 
                              left_on =['b_number', 'kegg_id'], 
                              right_on = ['gene', 'reaction_id'],
                             how = 'left').drop(['gene', 'reaction_id'], axis=1).rename({'b_number':'gene'}, axis=1)
eco_enzymes_merged

,rxn_id,kegg_id,rhea_id,Reactants,Products,EC,GPR,gene,enzyme_id,gene_id,ec_number,compound,kcat_values
0,2MAHMP,NaN,"[27915, 27914, 27916, 27917]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949,NaN,NaN,NaN
1,2MAHMP,NaN,"[27915, 27914, 27916, 27917]","[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949,NaN,NaN,NaN
2,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.-,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949,NaN,NaN,NaN
3,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.-,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949,NaN,NaN,NaN
4,TDP,R00615,"[27998, 28000, 28001, 27999]","[C00068, [C00001, C01328, C00001;C01328;D00001...","[C00080, C01081, [C00009, C00009;D05467]]",3.6.1.15,lcl_NC_006958_1_prot_WP_003855288_1_2949,lcl_NC_006958_1_prot_WP_003855288_1_2949,Q8NL63,lcl_NC_006958_1_prot_WP_003855288_1_2949,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
699,CAT,R00009,"[20309, 20310, 20311, 20312]","[[C00027, C00027;D00008]]","[[C00001, C01328, C00001;C01328;D00001;D03703;...",1.11.1.6,WP_011013509_1,WP_011013509_1,NaN,WP_011013509_1,NaN,NaN,NaN
700,SUCDi,NaN,"[29187, 29188, 29189, 29190]",[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013606_1,NaN,WP_011013606_1,NaN,NaN,NaN
701,SUCDi,NaN,"[29187, 29188, 29189, 29190]",[],[],NaN,WP_011013606_1 and WP_011013607_1,WP_011013607_1,NaN,WP_011013607_1,NaN,NaN,NaN
702,PYRt,NaN,NaN,[C00022],[C00022],NaN,WP_011014733_1,WP_011014733_1,NaN,WP_011014733_1,NaN,NaN,NaN


### 6. Extract directionalities and gapfill
Reactions which are associated with an enzyme, but not with a kcat from GotEnzymes will be assignes
a default kcat and if required a default molmass.

In [55]:
# Get directionalities and fill in the gaps
eco_enzymes_merged['direction'] = 'f'
                                              
for index, row in eco_enzymes_merged.iterrows():
    #there is a kegg id and kcat associated to this reaction
    if not isinstance(row.kcat_values, float):
        if row.compound in row.Products:
            eco_enzymes_merged.direction.iloc[index] = 'b'
        elif not row.compound in row.Reactants:
            print(f'Compound {row["compound"]} is not associated to reaction {row["kegg_id"]}')
            continue
    #there is no kcat associated to this reaction, try to use EC number to get a kcat value
    else: 
        got_enzyme_info = eco_enzymes_df[eco_enzymes_df['ec_number']==row.EC]
        for i, info in got_enzyme_info.iterrows():
            #is there any enzyme association if we look at the metabolites in the reaction and the EC number?
            if info.compound in row.Products:
                direction = 'b'
            elif info.compound in row.Reactants:
                direction = 'f'
            else:
                continue
            #change the kcat if it is not there already or create a new enzyme association
            if np.isnan(row.kcat_values):
                eco_enzymes_merged.direction.iloc[index] = 'b'
                eco_enzymes_merged.kcat_values.iloc[index] = info.kcat_values
            else:
                eco_enzymes_merged.loc[len(eco_enzymes_merged)] = row.to_list()[:-2]+[info.kcat_values, 'b']
        #add default information for unmappable proteins
        
        if (row.GPR == '') and not isinstance(row.EC, float): 
            eco_enzymes_merged.gene[index] = [f'Gene_{row.rxn_id}_{row.EC}']
            eco_enzymes_merged.GPR[index] = f'Gene_{row.rxn_id}_{row.EC}'
            eco_enzymes_merged.enzyme_id[index] = f'Enzyme_{row.rxn_id}_{row.EC}'
        if isinstance(row.gene, float) and len(row.GPR)>0:
            eco_enzymes_merged.gene[index] = [row.GPR]
        if not isinstance(row.enzyme_id, str) and len(row.GPR)>0:
            eco_enzymes_merged.enzyme_id[index] = f'Enzyme_{row.gene}'
        
        if np.isnan(row.kcat_values) & (isinstance(row.enzyme_id, str)) & (row.GPR!= 's0001'):
            eco_enzymes_merged.direction.iloc[index] = 'f'
            eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT
        if np.isnan(row.Mass) & (len(row.GPR)>2) & (row.GPR!= 's0001'):
            eco_enzymes_merged.Mass.iloc[index] = DEFAULT_PROT_MW
        
        
eco_enzymes_mapped = eco_enzymes_merged.drop(['ec_number', 'compound'], axis=1).rename({'Mass':'molMass',
                                                                                       'b_number': 'gene'}, axis =1)
eco_enzymes_mapped

/tmp/ipykernel_343616/253922216.py:41: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.direction.iloc[index] = 'f'
/tmp/ipykernel_343616/253922216.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  eco_enzymes_merged.kcat_values.iloc[index] = DEFAULT_KCAT


AttributeError: 'Series' object has no attribute 'Mass'

In [23]:
eco_enzymes_mapped.gene = [[g] for g in eco_enzymes_mapped.gene]
protein2gene, gene2protein = get_protein_gene_mapping(eco_enzymes_mapped, model)


# Ensure the enzyme complexes are merged on one row
eco_enzymes_mapped = merge_enzyme_complexes(eco_enzymes_mapped, gene2protein)

# Save the adjusted dataframe or continue processing
eco_enzymes_mapped

,rxn_id,kegg_id,rhea_id,mnx_id,Reactants,Products,EC,GPR,gene,enzyme_id,molMass,Length,kcat_values,direction
2,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,[lcl_NC_006958_1_prot_WP_003855288_1_2949],Enzyme_lcl_NC_006958_1_prot_WP_003855288_1_2949,39959.4825,NaN,NaN,f
3,2MAHMP,NaN,NaN,MNXR142616,"[[C00001, C01328, C00001;C01328;D00001;D03703;...","[C04556, C00080, [C00009, C00009;D05467]]",NaN,lcl_NC_006958_1_prot_WP_003855288_1_2949 or cg...,[cg3199],Enzyme_cg3199,39959.4825,NaN,NaN,f
9,3MBt2pp,NaN,NaN,MNXR94916,"[C00080, C08262]","[C08262, C00080]",NaN,cg0953,[cg0953],Q8NMS0,39959.4825,NaN,13.7,f
10,3MBt4pp,NaN,NaN,MNXR94917,"[C01330, C08262]","[C01330, C08262]",NaN,cg0953,[cg0953],Q8NS46,39959.4825,NaN,13.7,f
12,3OXCOAT,R00829,"[19482, 19483, 19481, 19484]",MNXR94967,"[C02232, C00010]","[C00091, C00024]",2.3.1.174,cg2625,[cg2625],Enzyme_cg2625,39959.4825,NaN,NaN,f
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2439,ZNabcpp,NaN,"[29795, 29796, 29797, 29798]",MNXR105281,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",3.6.3.5,cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,[cg2912],Enzyme_cg2912,39959.4825,NaN,NaN,f
2440,ZNabcpp,NaN,"[29795, 29796, 29797, 29798]",MNXR105281,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",3.6.3.5,cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,"[cg0042, cg0043]",Enzyme_cg0042,39959.4825,NaN,NaN,f
2441,ZNabcpp,NaN,"[29795, 29796, 29797, 29798]",MNXR105281,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",3.6.3.5,cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,"[cg0042, cg0043]",Enzyme_cg0043,39959.4825,NaN,NaN,f
2442,ZNabcpp,NaN,"[29795, 29796, 29797, 29798]",MNXR105281,"[[C00002, C00002;D08646], [C00001, C01328, C00...","[C00038, [C00008, C00008;G11113], C00080, [C00...",3.6.3.5,cg1040 or cg2912 or (cg0042 and cg0043) or (cg...,"[cg0042, cg0043]",Enzyme_cg0042,39959.4825,NaN,NaN,f


In [24]:
#save the dataframe
eco_enzymes_mapped = eco_enzymes_mapped.drop_duplicates(subset = ['rxn_id', 'enzyme_id', 'direction'],
                                                       ignore_index=True)
eco_enzymes_mapped.to_excel(AE_OUTPUT_FILE_PATH)

### 7. Save the dataframe to the proper format for building PAMs

In [25]:
# Get the information about the enzyme sectors
unused_enzymes_df = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = 'ExcessEnzymes')
translational_protein_df = pd.read_excel(PAM_DATA_FILE_PATH,
                                 sheet_name = 'Translational')

In [26]:
# Save it to a new excel file
with pd.ExcelWriter(OUTPUT_FILE_PATH) as writer:
    eco_enzymes_mapped.to_excel(writer, sheet_name='ActiveEnzymes')
    translational_protein_df.to_excel(writer, sheet_name='Translational')
    unused_enzymes_df.to_excel(writer, sheet_name = 'UnusedEnzyme')